In [1]:
%pwd

'/Users/khushishah/Documents/Projects/ML/AI-Medical-Chatbot/research'

In [2]:
import os
os.chdir("../")

In [3]:
%pwd

'/Users/khushishah/Documents/Projects/ML/AI-Medical-Chatbot'

In [6]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [9]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob='**/*.pdf',
        loader_cls= PyPDFLoader
    )

    documents = loader.load()
    return documents


In [10]:
extracted_data = load_pdf_files("data")

In [11]:
extracted_data

[Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'data/Medical_book.pdf', 'total_pages': 637, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'data/Medical_book.pdf', 'total_pages': 637, 'page': 1, 'page_label': '2'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION'),
 Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'data/Medical_book.pdf', 'total_pages': 637, 'page': 2, 'page_label': '3'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\nA-B\n1'),
 Doc

In [12]:
len(extracted_data)

637

In [15]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [16]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [17]:
minimal_docs

[Document(metadata={'source': 'data/Medical_book.pdf'}, page_content=''),
 Document(metadata={'source': 'data/Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION'),
 Document(metadata={'source': 'data/Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\nA-B\n1'),
 Document(metadata={'source': 'data/Medical_book.pdf'}, page_content='STAFF\nJacqueline L. Longe, Project Editor\nDeirdre S. Blanchfield, Associate Editor\nChristine B. Jeryan, Managing Editor\nDonna Olendorf, Senior Editor\nStacey Blachford, Associate Editor\nKate Kretschmann, Melissa C. McDade, Ryan\nThomason, Assistant Editors\nMark Springer, Technical Specialist\nAndrea Lopeman, Programmer/Analyst\nBarbara J. Yarrow,Manager, Imaging and Multimedia\nContent\nRobyn V . Young,Project Manager, Imaging and\nMultimedia Content\nDean Dauphinais, Senior Editor, Imaging and\nMultimedia C

In [18]:
#chunking

def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )

    texts = text_splitter.split_documents(minimal_docs)
    return texts

In [ ]:
text = text_split(minimal_docs)
print(f"Number of chunks: {len(text)}")

Number of chunks: 5860


In [22]:
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()

/var/folders/dk/6qmqdbk95njbg04q2r0w07jr0000gn/T/ipykernel_56813/2533971096.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3096.71it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
vector = embedding.embed_query("Hello World")

In [24]:
vector

[-0.03447726368904114,
 0.031023191288113594,
 0.006734950002282858,
 0.026109006255865097,
 -0.03936203196644783,
 -0.16030247509479523,
 0.06692396104335785,
 -0.006441459991037846,
 -0.04745053872466087,
 0.014758896082639694,
 0.070875383913517,
 0.05552759766578674,
 0.019193314015865326,
 -0.026251304894685745,
 -0.010109543800354004,
 -0.02694052830338478,
 0.022307414561510086,
 -0.022226668894290924,
 -0.14969265460968018,
 -0.01749309152364731,
 0.007676207460463047,
 0.0543522983789444,
 0.0032544347923249006,
 0.03172598406672478,
 -0.08462139219045639,
 -0.029406001791357994,
 0.05159563943743706,
 0.04812401533126831,
 -0.0033148108050227165,
 -0.058279164135456085,
 0.04196929186582565,
 0.02221071347594261,
 0.128188818693161,
 -0.02233891934156418,
 -0.011656287126243114,
 0.06292840093374252,
 -0.032876305282115936,
 -0.09122607856988907,
 -0.03117530420422554,
 0.052699532359838486,
 0.047034844756126404,
 -0.0842030718922615,
 -0.030056161805987358,
 -0.020744832232

In [25]:
print(f"len: {len(vector)}")

len: 384


In [102]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

True

In [103]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [104]:
OPENAI_API_KEY

'sk-mnop5678mnop5678mnop5678mnop5678mnop5678'

In [105]:
os.environ['PINECONE_API_KEY'] = PINECONE_API_KEY
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

In [106]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY


pc = Pinecone(api_key=pinecone_api_key)

In [34]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws",region="us-east-1")
    )

index = pc.Index(index_name)


In [36]:
from langchain_pinecone import PineconeVectorStore


docsearch = PineconeVectorStore.from_documents(
    documents = text,
    embedding=embedding,
    index_name=index_name
)



In [37]:
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [38]:
docsearch

In [47]:
# Adding Vol 2


loader = PyPDFLoader("data/Medical_book2.pdf")
extracted_book2 = loader.load()

In [48]:
extracted_book2

[Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:16:32-05:00', 'moddate': '2004-12-18T16:35:04-06:00', 'source': 'data/Medical_book2.pdf', 'total_pages': 759, 'page': 0, 'page_label': '1'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION'),
 Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:16:32-05:00', 'moddate': '2004-12-18T16:35:04-06:00', 'source': 'data/Medical_book2.pdf', 'total_pages': 759, 'page': 1, 'page_label': '2'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\nC-F\n2'),
 Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:16:32-05:00', 'moddate': '2004-12-18T16:35:04-06:00', 'source': 'data/Medical_book2.pdf', 'total_pages': 759, 'page': 2, 'page_label': '3'}, page_content='STAFF

In [49]:
minimal_docs2 = filter_to_minimal_docs(extracted_book2)

In [51]:
text2 = text_split(minimal_docs2)
print(f"Number of chunks: {len(text)}")

Number of chunks: 7024


In [52]:
# Connect to existing index
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)


# Append Book 2 embeddings
docsearch.add_documents(text2)

['17b957c6-80d0-45f2-8c5d-e0a1edf73902',
 '92e1b4f9-b54f-40c5-88f9-83752b0db984',
 'cd9ee1ce-1e40-4a6f-8ea2-5d4e67ca4490',
 '51a437f4-4e52-4b86-8f1d-50b60aa3426d',
 '4399f522-b882-4c3c-9d71-315374f6c36c',
 '641d0116-0dc4-446c-a9a5-a45c9f4d0f23',
 'cd48a281-55b2-4485-9c68-7cb729525697',
 'c05c7da2-9d2d-4b4c-a10d-09b4dedc24a8',
 '4667b299-0c22-4b37-817d-27124adb5f13',
 '26d3071a-520f-410f-9246-169030ebd70f',
 '5a7d6aef-f122-4c85-a9fc-60504103be00',
 '0d519480-b308-4956-93df-7b0d9e17e102',
 'ff874a4c-5157-4908-b69b-d68e9d519495',
 '3c185666-3492-422f-9cb6-77fe9b8b9464',
 'f9e915e2-242e-4b2a-a785-9eaa6a96cabc',
 '786ebe04-f4cc-4137-ac5b-7a7506e87484',
 'b489f48d-5446-40ff-a626-da5b2b6a3678',
 '57d94b78-2349-49fb-a420-aef7c20be830',
 'c48e89d4-6cce-42cc-8f01-7dfc177726e3',
 '55ec5350-0d0b-4001-b1a2-4542b865104d',
 '8ac0df90-cb73-4581-8bef-56d90c9d78d1',
 '567f3cfd-5f28-4715-8711-6ed8c08534d9',
 'e266651d-9ad4-4b1d-a5fa-bf1b6ba5dc30',
 'f3f954b2-5734-4833-9a52-6fa2da4597f5',
 'c3cf2601-471a-

In [53]:
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '189',
                                    'content-type': 'application/json',
                                    'date': 'Wed, 28 Jan 2026 19:52:30 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '5',
                                    'x-pinecone-request-id': '1004984310964194491',
                                    'x-pinecone-request-latency-ms': '4',
                                    'x-pinecone-response-duration-ms': '6'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 12884}},
 'storageFullness': 0.0,
 'total_vector_count': 12884,
 'vector_type': 'dense'}

In [54]:
retriever = docsearch.as_retriever(search_type='similarity', search_kwags={"k":3})

In [55]:
retrieved_docs = retriever.invoke("What is Acne?")

In [56]:
retrieved_docs

[Document(id='4940de15-72e7-4d28-bdee-b0ccff2cd260', metadata={'source': 'data/Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='745acbf6-0ffd-4884-8afa-ef028b824168', metadata={'source': 'data/Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'),
 Document(id='764b91d2-0d18-4970-803e-973dd8c15ed6', metadata={'source': 'data/Medical_book.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged with 

In [108]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(
    model = 'gpt-4o'
)

In [109]:
from langchain_core.runnables import Runnable
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI



In [110]:
system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [111]:
rag_chain = (
    {
        "context": retriever,
        "input": RunnablePassthrough()
    }
    | prompt
    | chatModel
    | StrOutputParser()
)


In [ ]:
rag_chain.invoke("What is diabetic retinopathy?")